Install reqs

In [ ]:
!pip install -q meeko prody rdkit vina gemmi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.0/294.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.5/37.5 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.1/103.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.8 MB/s eta 0:00:00


In [ ]:
!pip install py3Dmol

In [ ]:
# Get our protein
!wget https://files.rcsb.org/download/6LU7.pdb

--2026-04-25 14:51:35--  https://files.rcsb.org/download/6LU7.pdb
Resolving files.rcsb.org (files.rcsb.org)... 18.64.236.33, 18.64.236.48, 18.64.236.108, ...
Connecting to files.rcsb.org (files.rcsb.org)|18.64.236.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [text/plain]
Saving to: ‘6LU7.pdb’

6LU7.pdb                [ <=>                ] 233.51K  --.-KB/s    in 0.09s   

2026-04-25 14:51:36 (2.61 MB/s) - ‘6LU7.pdb’ saved [239112]



Prepare our protein

In [ ]:
from prody import parsePDB, writePDB

protein = parsePDB("6LU7.pdb")

# Keep protein chain A, remove waters and non-protein HETATM ligands
receptor = protein.select("chain A and protein")

writePDB("6LU7_receptor.pdb", receptor)

@> 2500 atoms and 1 coordinate set(s) were parsed in 0.05s.
DEBUG:.prody:2500 atoms and 1 coordinate set(s) were parsed in 0.05s.


'6LU7_receptor.pdb'

In [ ]:
!mk_prepare_receptor.py -i 6LU7_receptor.pdb -o 6LU7_receptor -p -v \
  --box_center -9.997 13.086 67.796 \
  --box_size 24 24 24

@> 2367 atoms and 1 coordinate set(s) were parsed in 0.03s.

Files written:
  6LU7_receptor.pdbqt <-- static (i.e., rigid) receptor input file
6LU7_receptor.box.txt <-- Vina-style box dimension file
6LU7_receptor.box.pdb <-- PDB file to visualize the grid box


Get the molecules we will screen

In [ ]:
from vina import Vina
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, Crippen
import pandas as pd
import os

molecules = {
    # Strong Mpro / coronavirus protease references
    "Nirmatrelvir": "CC1([C@@H]2[C@H]1[C@H](N(C2)C(=O)[C@H](C(C)(C)C)NC(=O)C(F)(F)F)C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)C",
    "Ensitrelvir": "Cn1cnc(CN2C(=O)N(Cc3cc(F)c(F)cc3F)C(=N\\c3cc4cn(C)nc4cc3Cl)\\NC2=O)n1",
    "Simnotrelvir": "CC(C)(C)[C@@H](C(=O)N1CC2(C[C@H]1C(=O)N[C@@H](C[C@@H]3CCNC3=O)C#N)SCCS2)NC(=O)C(F)(F)F",
    "GC376": "CC(C)C[C@@H](C(=O)N[C@@H](CC1CCNC1=O)C(O)S(=O)(=O)O)NC(=O)OCC2=CC=CC=C2",
    "GC373": "CC(C)C[C@@H](C(=O)N[C@@H](CC1CCNC1=O)C=O)NC(=O)OCC2=CC=CC=C2",
    "PF00835231": "CC(C)C[C@@H](C(=O)N[C@@H](C[C@@H]1CCNC1=O)C(=O)CO)NC(=O)C2=CC3=C(N2)C=CC=C3OC",

    # Smaller literature / repurposing-style inhibitors
    "Ebselen": "O=C1Nc2ccccc2[Se]1",
    "Carmofur": "CCCCCCNC(=O)Nc1cc(=O)[nH]c(=O)[nH]1",
    "Tideglusib": "O=C1N(c2ccccc2)N(Cc3ccccc3)C(=O)S1",
    "PX12": "CCC(C)SSC1=NC=CN1",

    # Natural compounds
    "Baicalein": "O=C1C=C(Oc2cc(O)c(O)c(O)c12)c3ccccc3",
    "Boceprevir": "O=C(N3[C@H](C(=O)NC(C(=O)C(=O)N)CC1CCC1)[C@H]2C(C)([C@H]2C3)C)[C@@H](NC(=O)NC(C)(C)C)C(C)(C)C",
    "Quercetin": "C1=CC(=C(C=C1C2=C(C(=O)C3=C(C=C(C=C3O2)O)O)O)O)O",
    "Curcumin": "COC1=C(C=CC(=C1)/C=C/C(=O)CC(=O)/C=C/C2=CC(=C(C=C2)OC)O)O",
    "EGCG": "C1[C@@H]([C@H](OC2=CC(=CC(=C21)O)O)C3=CC(=C(C(=C3)O)O)O)OC(=O)C4=CC(=C(C(=C4)O)O)O",

    # Experimental / open-science-like compounds
    "MI09": "O=C(NCc1cc(ccc1)C#N)Cc2cc3c(cccc3)nc2",
    "MI30": "O=C(NCc1cccc(C#N)c1)Cc2cc3c(c(Cl)ccc3)nc2",
    "ASAP0017445": "O=C(N(CC1=CN(C)N=N1)C[C@@]23C(N(C4=CN=CC5=C4C=CC=C5)C[C@@H]3CNC(C)C)=O)C6=C2C=C(Cl)C=C6",

    "Ibuzatrelvir": "CC(C)(C)[C@@H](C(=O)N1C[C@@H](C[C@H]1C(=O)N[C@@H](C[C@@H]2CCNC2=O)C#N)C(F)(F)F)NC(=O)OC",
    "Masitinib mesylate": "CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC(=CS4)C5=CN=CC=C5.CS(=O)(=O)O",
    "Atazanavir": "CC(C)(C)[C@@H](C(=O)N[C@@H](CC1=CC=CC=C1)[C@H](CN(CC2=CC=C(C=C2)C3=CC=CC=N3)NC(=O)[C@H](C(C)(C)C)NC(=O)OC)O)NC(=O)OC",
    "AG7404": "CCOC(=O)/C=C/[C@H](C[C@@H]1CCNC1=O)NC(=O)[C@H](CC#C)N2C=CC=C(C2=O)NC(=O)C3=NOC(=C3)C",
    "Rupintrivir": "CCOC(=O)/C=C/[C@H](C[C@@H]1CCNC1=O)NC(=O)[C@H](CC2=CC=C(C=C2)F)CC(=O)[C@H](C(C)C)NC(=O)C3=NOC(=C3)C",
    "Pimozide": "C1CN(CCC1N2C3=CC=CC=C3NC2=O)CCCC(C4=CC=C(C=C4)F)C5=CC=C(C=C5)F",
    "Ebastine": "CC(C)(C)C1=CC=C(C=C1)C(=O)CCCN2CCC(CC2)OC(C3=CC=CC=C3)C4=CC=CC=C4",
    "Bepridil hydrochloride": "CC(C)COCC(CN(CC1=CC=CC=C1)C2=CC=CC=C2)N3CCCC3.Cl",
    "Simeprevir": "CC1=C(C=CC2=C1N=C(C=C2O[C@@H]3C[C@@H]4[C@@H](C3)C(=O)N(CCCC/C=C\\[C@@H]5C[C@]5(NC4=O)C(=O)NS(=O)(=O)C6CC6)C)C7=NC(=CS7)C(C)C)OC",
    "Tipranavir": "CCC[C@]1(CC(=C(C(=O)O1)[C@H](CC)C2=CC(=CC=C2)NS(=O)(=O)C3=NC=C(C=C3)C(F)(F)F)O)CCC4=CC=CC=C4",
    "Nelfinavir": "CC1=C(C=CC=C1O)C(=O)N[C@@H](CSC2=CC=CC=C2)[C@@H](CN3C[C@H]4CCCC[C@H]4C[C@H]3C(=O)NC(C)(C)C)O"
}

Prepare the molecules for screening

In [ ]:
rows = []

for name, smiles in molecules.items():
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        print(f"Invalid SMILES for {name}")
        continue
    mol = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    # params.maxAttempts = 1000
    params.useRandomCoords = True

    conf_id = AllChem.EmbedMolecule(mol, params)
    if (conf_id == -1):
      print(f"Skipping {name} due to issue embedding molecule")
    AllChem.MMFFOptimizeMolecule(mol)
    Chem.MolToMolFile(mol, f"{name}.sdf")
    command = f"mk_prepare_ligand.py -i {name}.sdf -o {name}.pdbqt"
    os.system(command)
    rows.append({
        "name": name,
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol),
        "Rotatable Bonds": Descriptors.NumRotatableBonds(mol),
        "Heavy Atoms": mol.GetNumHeavyAtoms()
    })
    print(f"Prepared {name}")

Prepared Nirmatrelvir
Prepared Ensitrelvir
Prepared Simnotrelvir
Prepared GC376
Prepared GC373
Prepared PF00835231


[15:24:42] UFFTYPER: Unrecognized atom type: Se2+2 (9)
[15:24:42] UFFTYPER: Unrecognized atom type: Se2+2 (9)


Prepared Ebselen
Prepared Carmofur
Prepared Tideglusib
Prepared PX12
Prepared Baicalein
Prepared Boceprevir
Prepared Quercetin
Prepared Curcumin
Prepared EGCG
Prepared MI09
Prepared MI30
Prepared ASAP0017445
Prepared Ibuzatrelvir
Prepared Masitinib mesylate
Prepared Atazanavir
Prepared AG7404
Prepared Rupintrivir
Prepared Pimozide
Prepared Ebastine
Prepared Bepridil hydrochloride
Prepared Simeprevir
Prepared Tipranavir
Prepared Nelfinavir


Do a light initial pass for screening

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed

RECEPTOR = "6LU7_receptor.pdbqt"
CENTER = [-10.7, 12.4, 68.8]
BOX = [22, 22, 22]

def dock_one(name, exhaustiveness=2, n_poses=1):
    try:
        print(f"Docking {name}...")

        v = Vina(sf_name="vina")
        v.set_receptor(RECEPTOR)
        v.set_ligand_from_file(f"{name}.pdbqt")
        v.compute_vina_maps(center=CENTER, box_size=BOX)

        v.dock(exhaustiveness=exhaustiveness, n_poses=n_poses)
        score = v.energies(n_poses=n_poses)[0][0]

        v.write_poses(f"{name}_docked.pdbqt", n_poses=n_poses, overwrite=True)

        return {
            "molecule": name,
            "vina_score_kcal_mol": score,
            # "ligand_efficiency": score/
            "status": "success"
        }

    except Exception as e:
        return {
            "molecule": name,
            "vina_score_kcal_mol": None,
            # "ligand_efficiency": None,
            "status": f"failed: {e}"
        }

print("Started initial screening...")

results = []

with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(dock_one, name) for name in molecules]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(result["molecule"], result["status"])

df = pd.DataFrame(results).sort_values("vina_score_kcal_mol").merge(pd.DataFrame(rows), left_on="molecule", right_on="name")
df

Started initial screening...
Docking Nirmatrelvir...
Docking Ensitrelvir...
Docking Simnotrelvir...
Ensitrelvir success
Docking GC376...
Nirmatrelvir success
Docking GC373...
Simnotrelvir success
Docking PF00835231...
GC376 success
Docking Ebselen...
GC373 success
Docking Carmofur...
Ebselen failed: Error: file Ebselen.pdbqt does not exist.
Docking Tideglusib...
Carmofur success
Docking PX12...
Tideglusib success
Docking Baicalein...
PX12 success
Docking Boceprevir...
Baicalein success
Docking Quercetin...
PF00835231 success
Docking Curcumin...
Quercetin success
Docking EGCG...
Curcumin success
Docking MI09...
Boceprevir success
Docking MI30...
MI09 success
Docking ASAP0017445...
EGCG success
Docking Ibuzatrelvir...
MI30 success
Docking Masitinib mesylate...
Docking Atazanavir...
ASAP0017445 success
Masitinib mesylate failed: Error: file Masitinib mesylate.pdbqt does not exist.
Docking AG7404...
Ibuzatrelvir success
Docking Rupintrivir...
AG7404 success
Docking Pimozide...
Rupintrivir 

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms
0,Ensitrelvir,-9.305,success,Ensitrelvir,531.886,3.60760,1,6,5,37
1,EGCG,-9.119,success,EGCG,458.375,2.23320,8,11,11,33
2,ASAP0017445,-8.556,success,ASAP0017445,544.059,3.57150,1,6,6,39
3,Pimozide,-8.489,success,Pimozide,461.556,6.39340,1,2,7,34
4,GC376,-8.053,success,GC376,485.559,0.54470,5,8,13,33
5,PF00835231,-8.008,success,PF00835231,472.542,0.89340,5,6,12,34
6,Ibuzatrelvir,-7.914,success,Ibuzatrelvir,489.495,1.07108,3,6,6,34
7,Rupintrivir,-7.859,success,Rupintrivir,598.672,2.82492,3,8,15,43
8,AG7404,-7.731,success,AG7404,523.546,0.47482,3,8,11,38
9,Atazanavir,-7.685,success,Atazanavir,704.869,4.21160,5,9,15,51


Find our strongest matches

In [ ]:
df["ligand_efficiency"] = df["vina_score_kcal_mol"] / df["Heavy Atoms"]
df = df.sort_values(["vina_score_kcal_mol", "ligand_efficiency"])
df

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
0,Ensitrelvir,-9.305,success,Ensitrelvir,531.886,3.60760,1,6,5,37,-0.251486
1,EGCG,-9.119,success,EGCG,458.375,2.23320,8,11,11,33,-0.276333
2,ASAP0017445,-8.556,success,ASAP0017445,544.059,3.57150,1,6,6,39,-0.219385
3,Pimozide,-8.489,success,Pimozide,461.556,6.39340,1,2,7,34,-0.249676
4,GC376,-8.053,success,GC376,485.559,0.54470,5,8,13,33,-0.244030
5,PF00835231,-8.008,success,PF00835231,472.542,0.89340,5,6,12,34,-0.235529
6,Ibuzatrelvir,-7.914,success,Ibuzatrelvir,489.495,1.07108,3,6,6,34,-0.232765
7,Rupintrivir,-7.859,success,Rupintrivir,598.672,2.82492,3,8,15,43,-0.182767
8,AG7404,-7.731,success,AG7404,523.546,0.47482,3,8,11,38,-0.203447
9,Atazanavir,-7.685,success,Atazanavir,704.869,4.21160,5,9,15,51,-0.150686


In [ ]:
strong_candidates = df[(df["vina_score_kcal_mol"] <= -7.5) & (df["ligand_efficiency"] <= -0.2)]

In [ ]:
strong_candidates

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
0,Ensitrelvir,-9.305,success,Ensitrelvir,531.886,3.60760,1,6,5,37,-0.251486
1,EGCG,-9.119,success,EGCG,458.375,2.23320,8,11,11,33,-0.276333
2,ASAP0017445,-8.556,success,ASAP0017445,544.059,3.57150,1,6,6,39,-0.219385
3,Pimozide,-8.489,success,Pimozide,461.556,6.39340,1,2,7,34,-0.249676
4,GC376,-8.053,success,GC376,485.559,0.54470,5,8,13,33,-0.244030
5,PF00835231,-8.008,success,PF00835231,472.542,0.89340,5,6,12,34,-0.235529
6,Ibuzatrelvir,-7.914,success,Ibuzatrelvir,489.495,1.07108,3,6,6,34,-0.232765
8,AG7404,-7.731,success,AG7404,523.546,0.47482,3,8,11,38,-0.203447
10,Baicalein,-7.601,success,Baicalein,270.240,2.41960,3,5,4,20,-0.380050
11,Quercetin,-7.555,success,Quercetin,302.238,2.01090,5,7,6,22,-0.343409


Perform a more stringent second round

In [ ]:
results = []

with ProcessPoolExecutor(max_workers=2) as executor:
    futures = [executor.submit(dock_one, name, exhaustiveness=16, n_poses=10) for name in strong_candidates["molecule"]]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)
        print(result["molecule"], result["status"])

df_stage2 = pd.DataFrame(results).sort_values("vina_score_kcal_mol").merge(pd.DataFrame(rows), left_on="molecule", right_on="name")
df_stage2

Docking Ensitrelvir...Docking EGCG...

Docking ASAP0017445...
Ensitrelvir success
Docking Pimozide...
EGCG success
Docking GC376...
ASAP0017445 success
Docking PF00835231...
Pimozide success
Docking Ibuzatrelvir...
GC376 success
Docking AG7404...
PF00835231 success
Docking Baicalein...
Ibuzatrelvir success
Docking Quercetin...
Baicalein success
Docking MI30...
Quercetin success
AG7404 success
MI30 success


,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms
0,Ensitrelvir,-9.322,success,Ensitrelvir,531.886,3.60760,1,6,5,37
1,EGCG,-9.126,success,EGCG,458.375,2.23320,8,11,11,33
2,Pimozide,-8.789,success,Pimozide,461.556,6.39340,1,2,7,34
3,ASAP0017445,-8.618,success,ASAP0017445,544.059,3.57150,1,6,6,39
4,PF00835231,-8.233,success,PF00835231,472.542,0.89340,5,6,12,34
5,GC376,-8.077,success,GC376,485.559,0.54470,5,8,13,33
6,AG7404,-8.070,success,AG7404,523.546,0.47482,3,8,11,38
7,Ibuzatrelvir,-7.934,success,Ibuzatrelvir,489.495,1.07108,3,6,6,34
8,Baicalein,-7.717,success,Baicalein,270.240,2.41960,3,5,4,20
9,Quercetin,-7.555,success,Quercetin,302.238,2.01090,5,7,6,22


In [ ]:
df_stage2["ligand_efficiency"] = df_stage2["vina_score_kcal_mol"] / df_stage2["Heavy Atoms"]
df_stage2 = df_stage2.sort_values(["vina_score_kcal_mol", "ligand_efficiency"])
strong_candidates_2 = df_stage2[
    (df_stage2["vina_score_kcal_mol"] <= -7.8) &
    (df_stage2["ligand_efficiency"] <= -0.23) &
    (df_stage2["MW"] <= 550) &
    (df_stage2["Rotatable Bonds"] <= 12) &
    (df_stage2["Heavy Atoms"] <= 40) &
    (df_stage2["HBD"] <= 6) &
    (df_stage2["HBA"] <= 10) &
    (df_stage2["LogP"] <= 5.0)
].copy()

In [ ]:
strong_candidates_2

,molecule,vina_score_kcal_mol,status,name,MW,LogP,HBD,HBA,Rotatable Bonds,Heavy Atoms,ligand_efficiency
0,Ensitrelvir,-9.322,success,Ensitrelvir,531.886,3.60760,1,6,5,37,-0.251946
4,PF00835231,-8.233,success,PF00835231,472.542,0.89340,5,6,12,34,-0.242147
7,Ibuzatrelvir,-7.934,success,Ibuzatrelvir,489.495,1.07108,3,6,6,34,-0.233353
